In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("amazon_india_complete_2015_2025.csv")
df.head()

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating
0,TXN_2023_00063013,2023-07-23,CUST_2023_00018393,PROD_000454,Vivo Y95 64GB Black,Electronics,Smartphones,Vivo,27340.84,21.57,...,False,NaN,4.0,Delivered,7,2023,3,0.20,True,3.5
1,TXN_2021_00064486,20-07-2021,CUST_2015_00002865,PROD_000579,Realme Realme 3 128GB Black,Electronics,Smartphones,Realme,32907.49,0.00,...,False,NaN,5/5,Delivered,7,2021,3,0.21,False,4.5
2,TXN_2017_00065617,2017-11-16,CUST_2016_00004057,PROD_000295,Vivo V7 32GB Blue,Electronics,Smartphones,Vivo,"47,052.18",21.91,...,False,NaN,5.0,Delivered,11,2017,4,0.24,True,4.3
3,TXN_2020_00054393,2020-05-04,CUST_2020_00014574,PROD_001654,Alienware Pavilion 4GB RAM Silver,Electronics,Laptops,Alienware,238725.44,59.60,...,True,Summer Sale,5.0 stars,Delivered,5,2020,2,1.85,Yes,3.6
4,TXN_2018_00071646,2018-10-09,CUST_2018_00006275,PROD_000095,Motorola Moto X Play 16GB White,Electronics,Smartphones,Motorola,25970.76,0.00,...,False,NaN,4.0,Delivered,10,2018,4,0.16,False,3.7


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1127609 entries, 0 to 1127608
Data columns (total 34 columns):
 #   Column                  Non-Null Count    Dtype  
---  ------                  --------------    -----  
 0   transaction_id          1127609 non-null  object 
 1   order_date              1127609 non-null  object 
 2   customer_id             1127609 non-null  object 
 3   product_id              1127609 non-null  object 
 4   product_name            1127609 non-null  object 
 5   category                1127609 non-null  object 
 6   subcategory             1127609 non-null  object 
 7   brand                   1127609 non-null  object 
 8   original_price_inr      1127609 non-null  object 
 9   discount_percent        1127609 non-null  float64
 10  discounted_price_inr    1127609 non-null  float64
 11  quantity                1127609 non-null  int64  
 12  subtotal_inr            1127609 non-null  float64
 13  delivery_charges        1037408 non-null  float64
 14  fi

In [3]:
df.isnull().sum()

transaction_id                 0
order_date                     0
customer_id                    0
product_id                     0
product_name                   0
category                       0
subcategory                    0
brand                          0
original_price_inr             0
discount_percent               0
discounted_price_inr           0
quantity                       0
subtotal_inr                   0
delivery_charges           90201
final_amount_inr               0
customer_city                  0
customer_state                 0
customer_tier                  0
customer_spending_tier         0
customer_age_group        135315
payment_method                 0
delivery_days                  0
delivery_type                  0
is_prime_member                0
is_festival_sale               0
festival_name             777736
customer_rating           341696
return_status                  0
order_month                    0
order_year                     0
order_quar

In [4]:
df.columns

Index(['transaction_id', 'order_date', 'customer_id', 'product_id',
       'product_name', 'category', 'subcategory', 'brand',
       'original_price_inr', 'discount_percent', 'discounted_price_inr',
       'quantity', 'subtotal_inr', 'delivery_charges', 'final_amount_inr',
       'customer_city', 'customer_state', 'customer_tier',
       'customer_spending_tier', 'customer_age_group', 'payment_method',
       'delivery_days', 'delivery_type', 'is_prime_member', 'is_festival_sale',
       'festival_name', 'customer_rating', 'return_status', 'order_month',
       'order_year', 'order_quarter', 'product_weight_kg', 'is_prime_eligible',
       'product_rating'],
      dtype='object')

Question 1
Your dataset contains order_date in multiple formats: 'DD/MM/YYYY', 'DD-MM-YY', 'YYYY-MM-DD', and some invalid entries like '32/13/2020'. Clean and standardize all dates to 'YYYY-MM-DD' format, handling invalid dates appropriately.


In [5]:
df['order_date'] = pd.to_datetime(
    df['order_date'], 
    errors='coerce',     # turn bad ones into NaT
    dayfirst=True,       # needed for DD-MM-YYYY and DD/MM/YYYY
    format="mixed"       # <-- NEW in Pandas 2.0, handles mixed styles
)


In [6]:
df.shape

(1127609, 34)

Question 2
The original_price_inr column contains mixed data types: numeric values, text with '₹' symbols, comma separators ('₹1,25,000'), and some entries like 'Price on Request'. Clean this column to contain only numeric values in Indian Rupees. 


In [7]:
df["original_price_inr"] = (
    df["original_price_inr"]
    .astype(str)                  
    .str.replace("₹", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.replace("Rs", "", regex=False)
)
df['original_price_inr'] = pd.to_numeric(
    df['original_price_inr'], errors='coerce'
)

df["original_price_inr"] = df["original_price_inr"].abs()


In [8]:
df.shape

(1127609, 34)

Question 3
Customer ratings appear in various formats: '5.0', '4 stars', '3/5', '2.5/5.0', and some missing values. Standardize all ratings to numeric scale 1.0-5.0, handling inconsistent formats and missing values strategically.


In [9]:
df['customer_rating'] = (
    df['customer_rating']
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace('stars', '', regex=False)
    .str.replace('star', '', regex=False)
)


In [10]:
def convert_fraction_rating(x):
    if '/' in x:
        try:
            num, den = x.split('/')
            return (float(num) / float(den)) * 5
        except:
            return np.nan
    return x


In [11]:
df['customer_rating'] = (
    df['customer_rating']
    .apply(convert_fraction_rating)
)


In [12]:
df['customer_rating'] = pd.to_numeric(
    df['customer_rating'],
    errors='coerce'
)


In [13]:
len(df)


1127609

In [14]:
missing_pct = (
    df['customer_rating'].isna().mean() * 100
)
print(f"Missing Ratings: {missing_pct:.2f}%")


Missing Ratings: 30.30%


In [15]:
df['customer_rating'] = (
    df
    .groupby('category')['customer_rating']
    .transform(lambda x: x.fillna(x.median()))
)


In [16]:
df.shape

(1127609, 34)

Question 4
The customer_city column has inconsistent naming: 'Bangalore/Bengaluru', 'Mumbai/Bombay', 'Delhi/New Delhi', along with spelling errors and case variations. Standardize all city names and handle geographical variations.


In [17]:
city_map = {
    # Bengaluru
    'bangalore': 'Bangalore',
    'bengalore': 'Bangalore',
    'BANGALORE': 'Bangalore',
    'banglore': 'Bangalore',
    'bengaluru': 'Bangalore',

    # Mumbai
    'mumbai': 'Mumbai',
    'mumba': 'Mumbai',
    'bombay': 'Mumbai',
    'Mumbai ': 'Mumbai',
    'Bombay': 'Mumbai',
    'MUMBAI': 'Mumbai',

    # Delhi
    'delhi': 'Delhi',
    'new delhi': 'Delhi',
    'delhi ncr': 'Delhi',
    'DELHI': 'Delhi',

    # Chennai
    'chennai': 'Chennai',
    'chenai': 'Chennai',
    'madras': 'Chennai',
    'CHENNAI': 'Chennai',
    'Madras': 'Chennai',
    'Chennai ': 'Chennai',

    # Kolkata
    'KOLKATA': 'Kolkata',
    'kolkata': 'Kolkata',
    'calcutta': 'Kolkata'

}
df['customer_city'] = (
    df['customer_city']
    .str.lower()
    .str.strip()
    .replace(city_map)
    .str.title()
)


In [18]:
df.shape

(1127609, 34)

Question 5
Boolean columns (is_prime_member, is_prime_eligible, is_festival_sale) contain mixed values: True/False, Yes/No, 1/0, Y/N, and some missing entries. Convert all boolean columns to consistent True/False format.



In [19]:
bool_map = {
    'yes': True, 'y': True, '1': True, 'true': True,
    'no': False, 'n': False, '0': False, 'false': False
}

bool_cols = ['is_prime_member', 'is_prime_eligible', 'is_festival_sale']

for col in bool_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.lower()
        .map(bool_map)
    )


In [20]:
df.shape

(1127609, 34)

Question 6
Product categories have variations: 'Electronics/Electronic/ELECTRONICS/Electronics & Accessories'. Standardize category names across the dataset and ensure consistent naming conventions.


In [21]:
category_map = {
    'electronics': 'Electronics',
    'electronic': 'Electronics',
    'electronicss': 'Electronics',
    'ELECTRONICS': 'Electronics',
    'electronics & accessories': 'Electronics'
}
df['category'] = (
    df['category']
    .str.lower()
    .str.strip()
    .replace(category_map)
    .str.title()
)


In [22]:
df.shape

(1127609, 34)

Question 7
The delivery_days column contains negative values, text entries like 'Same Day', '1-2 days', and some unrealistic values like 50 days. Clean this column to contain only valid numeric delivery days.


In [23]:
df['delivery_days'].value_counts()
#df["delivery_days"].isna().sum()

delivery_days
3           288488
1           210462
4           206438
5           136608
2           126280
6           102668
7            34116
-1            6836
1-2 days      4534
Same Day      4469
Express       2317
0             2218
15            2175
Name: count, dtype: int64

In [24]:
df['delivery_days'] = df['delivery_days'].replace({
    'Same Day': 0,
    '1-2 days': 2,
    'Express': 1,
    "-1": 0,
})




df['delivery_days'].isnull().sum()


np.int64(0)

In [25]:
df["delivery_days"] = pd.to_numeric(
    df["delivery_days"],
    errors="coerce"
)
df["delivery_days"] = df["delivery_days"].astype("Int64")

In [26]:
df['delivery_days'].value_counts()

delivery_days
3     288488
1     212779
4     206438
5     136608
2     130814
6     102668
7      34116
0      13523
15      2175
Name: count, dtype: Int64

In [27]:
df.shape

(1127609, 34)

Question 8
Identify and handle duplicate transactions where the same customer, product, date, and amount appear multiple times. Some duplicates are genuine (bulk orders) while others are data errors. Develop a strategy to distinguish and handle both cases.


In [28]:
df.duplicated().sum()
df.duplicated(subset=['transaction_id']).sum()

np.int64(0)

In [29]:
df = df[~df["transaction_id"].str.contains("_DUP", na=False)]

In [30]:
df.shape

(1122000, 34)

Question 9
The dataset contains outlier prices where some products show prices 100x higher than expected due to data entry errors (decimal point issues). Identify and correct these outliers using statistical methods and domain knowledge.


In [31]:
df["product_mean_price"] = (
    df.groupby("product_id")["original_price_inr"]
    .transform("mean")
) 

In [32]:
df["original_price_inr"].max()

np.float64(33371693.0)

In [33]:
df["product_mean_price"].isna().sum()

np.int64(0)

In [34]:
pd.set_option("display.max_rows", None)
print(df.loc[
    df["original_price_inr"] > 200000,
    ["product_name", "original_price_inr"]
])

                                    product_name  original_price_inr
3              Alienware Pavilion 4GB RAM Silver           238725.44
22           Apple iPhone 13 Pro Max 256GB White           392777.17
38              Alienware MacBook 4GB RAM Silver           227208.08
60                              LG 4K TV Premium           221436.39
105              Apple iPhone 13 mini 64GB White           213909.05
189                   HP Inspiron 4GB RAM Silver           206655.37
210              Apple iPhone XS Max 256GB Black           238086.62
248                    Apple iPhone 15 64GB Gold           262527.62
259                     Samsung Smart TV Premium           647713.50
274                    Apple iPhone 6 16GB White           238513.74
301                Apple iPhone 11 Pro 64GB Blue           245823.33
325                   Noise Sports Watch Premium          4631753.00
330                     Apple iPhone X 16GB Blue           238760.02
352              Apple iPhone XS M

In [35]:
threshold = 1.1


df["original_price_inr"] = df["original_price_inr"].where(
    df["original_price_inr"] <= threshold * df["product_mean_price"],
    df["product_mean_price"]
)

In [36]:
df["original_price_inr"].describe()

count    1.122000e+06
mean     6.550130e+04
std      5.052344e+04
min      1.067270e+03
25%      2.879170e+04
50%      4.614487e+04
75%      9.287530e+04
max      8.955562e+05
Name: original_price_inr, dtype: float64

In [37]:
pd.set_option("display.max_rows", None)
print(df.loc[
    df["original_price_inr"] > 200000,
    ["product_name", "original_price_inr"]
])

                                 product_name  original_price_inr
3           Alienware Pavilion 4GB RAM Silver       208663.542648
22        Apple iPhone 13 Pro Max 256GB White       309308.784240
60                           LG 4K TV Premium       221436.390000
105           Apple iPhone 13 mini 64GB White       213909.050000
210           Apple iPhone XS Max 256GB Black       238086.620000
248                 Apple iPhone 15 64GB Gold       262527.620000
274                 Apple iPhone 6 16GB White       238513.740000
301             Apple iPhone 11 Pro 64GB Blue       245823.330000
330                  Apple iPhone X 16GB Blue       238760.020000
352           Apple iPhone XS Max 256GB Black       238086.620000
373                Apple iPhone XR 128GB Blue       234876.440000
379        Apple iPhone 13 Pro Max 64GB Black       328204.910000
380                MSI Inspiron 4GB RAM Black       277299.570000
449                 Apple iPhone 6 16GB Black       224159.470000
462       

In [38]:
Q1 = df["original_price_inr"].quantile(0.25)
Q3 = df["original_price_inr"].quantile(0.75)

IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

In [39]:
upper_limit

np.float64(189000.70880503143)

In [40]:
mask = df["original_price_inr"] > upper_limit


df.loc[mask, "original_price_inr"] = df.loc[mask, "product_mean_price"]

df.drop(columns=["product_mean_price"], inplace=True)


In [41]:
df.shape

(1122000, 34)

Question 10
Payment methods contain inconsistent naming: 'UPI/PhonePe/GooglePay', 'Credit Card/CREDIT_CARD/CC', 'Cash on Delivery/COD/C.O.D'. Standardize payment method categories and create a clean categorical hierarchy.


In [42]:
df['payment_method'].value_counts()

payment_method
UPI            382356
COD            321262
Credit Card    171396
Debit Card     139490
Net Banking     64620
Wallet          22678
BNPL            20198
Name: count, dtype: int64

In [43]:
df.shape

(1122000, 34)

In [44]:
df['festival_name'] = df['festival_name'].fillna('No Festival')

In [45]:
df['delivery_charges'] = df['delivery_charges'].fillna(0)

In [46]:
df['customer_age_group'] = df['customer_age_group'].fillna('Unknown')

In [47]:
df["customer_rating"].value_counts()

customer_rating
4.5    597440
5.0    200797
4.0    197324
3.5     79756
3.0     46683
Name: count, dtype: int64

In [48]:
df['transaction_id'].nunique()

1122000

In [49]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1122000 entries, 0 to 1127608
Data columns (total 34 columns):
 #   Column                  Non-Null Count    Dtype         
---  ------                  --------------    -----         
 0   transaction_id          1122000 non-null  object        
 1   order_date              1122000 non-null  datetime64[ns]
 2   customer_id             1122000 non-null  object        
 3   product_id              1122000 non-null  object        
 4   product_name            1122000 non-null  object        
 5   category                1122000 non-null  object        
 6   subcategory             1122000 non-null  object        
 7   brand                   1122000 non-null  object        
 8   original_price_inr      1122000 non-null  float64       
 9   discount_percent        1122000 non-null  float64       
 10  discounted_price_inr    1122000 non-null  float64       
 11  quantity                1122000 non-null  int64         
 12  subtotal_inr       

In [50]:
df.to_csv("amazon_india_2015_to_2025_cleaned.csv", index=False)

In [51]:
df.dtypes

transaction_id                    object
order_date                datetime64[ns]
customer_id                       object
product_id                        object
product_name                      object
category                          object
subcategory                       object
brand                             object
original_price_inr               float64
discount_percent                 float64
discounted_price_inr             float64
quantity                           int64
subtotal_inr                     float64
delivery_charges                 float64
final_amount_inr                 float64
customer_city                     object
customer_state                    object
customer_tier                     object
customer_spending_tier            object
customer_age_group                object
payment_method                    object
delivery_days                      Int64
delivery_type                     object
is_prime_member                     bool
is_festival_sale

In [52]:
amazon_raw = pd.read_csv("amazon_india_complete_2015_2025.csv")

In [53]:
comparison = pd.DataFrame({
    'Before_Cleaning': amazon_raw.isna().sum(),
    'After_Cleaning': df.isna().sum()
})
comparison


,Before_Cleaning,After_Cleaning
transaction_id,0,0
order_date,0,0
customer_id,0,0
product_id,0,0
product_name,0,0
category,0,0
subcategory,0,0
brand,0,0
original_price_inr,0,0
discount_percent,0,0
